In [1]:
# Standard Python modules
import os, sys
import yaml
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import re
import seaborn as sns
import cartopy
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar # different way to handle colorbar
import matplotlib.ticker as mticker
import cmocean.cm as cmo
# cartopy
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
import cartopy.feature as cfeature

# extras
%matplotlib inline
import geopandas as gpd
import shapely.geometry

# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
from utils import roundPartial, select_months_ds, get_startmon_and_endmon
from plotter import draw_basemap, plot_terrain
from trajectory_post_funcs import calculate_heatmaps_from_trajectories
import customcmaps as ccmap
from load_shapefiles import load_region_shp, load_HUC8
from load_trajectories import load_trajectories_based_on_region

pd.options.display.float_format = "{:,.2f}".format # makes it so pandas tables display only first two decimals

ERROR 1: PROJ: proj_create_from_database: Open of /home/dnash/miniconda3/envs/SEAK-impacts/share/proj failed


In [2]:
path_to_data = '/expanse/nfs/cw3e/cwp140/' 
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write
path_to_figs = '../figs/'      # figures

In [3]:
HUC8_ID_lst = [14050001, ## upper yampa
               14010001, ## roaring fork
               14020002, ## upper gunnison
               14080101, ## upper san juan
               # 14050005, ## upper white
               # 14050002, ## lower yampa
               # 14080104, ## animas (San Juans),
               # 14030002, ## upper dolores
               # 11020002, ## arkansas - Pueblo Reservoir
               # 10190005 ## St. Vrain (Boulder)
               # 14030005, ## 'Upper Colorado-Kane Springs'
               # 10190002, ## 'Upper South Platte'
               # 10190018, ## 'Lower South Platte'
               # 10190012, ## 'Middle South Platte-Sterling'
               # 11020001 ## Arkansas Headwaters
               # 11020009 ## Upper Arkansas-John Martin Reservoir
              ]

In [4]:
## load watershed shapefile and predefined regions shapefile
polys = load_HUC8()
plot_poly = load_region_shp(polys)

# ## if you are plotting individual subbasins use this
# plot_poly = polys[(polys.HUC8 == str(HUC8_ID_lst[i]))]
# plot_poly = regions

## load all trajectories categorized by region
ds = load_trajectories_based_on_region()
ds

<xarray.Dataset>
Dimensions:             (index: 72, start_date: 1641, HUC8: 92)
Coordinates:
  * index               (index) int64 0 1 2 3 4 5 6 7 ... 65 66 67 68 69 70 71
  * start_date          (start_date) datetime64[ns] 2000-01-11 ... 2023-12-26
  * HUC8                (HUC8) object '14050002' '14040109' ... '11030001'
    time                (HUC8, start_date, index) datetime64[ns] NaT NaT ... NaT
    lon                 (HUC8, start_date, index) float64 nan nan ... nan nan
    lat                 (HUC8, start_date, index) float64 nan nan ... nan nan
    region              (HUC8) <U17 'northern_upper_CO' ... 'eastern_CO'
Data variables: (12/21)
    level               (HUC8, start_date, index) float64 nan nan ... nan nan
    q                   (HUC8, start_date, index) float64 nan nan ... nan nan
    u                   (HUC8, start_date, index) float64 nan nan ... nan nan
    v                   (HUC8, start_date, index) float64 nan nan ... nan nan
    w                   (HUC8, start_date, index) float64 nan nan ... nan nan
    IVT                 (HUC8, start_date, index) float64 nan nan ... nan nan
    ...                  ...
    tARget_strict       (HUC8, start_date) float64 nan nan nan ... nan nan nan
    coastal_IVT_strict  (HUC8, start_date) float64 nan nan nan ... nan 46.89 nan
    time_match          (HUC8, start_date) object nan nan ... nan
    lev_match           (HUC8, start_date) float64 nan nan nan ... nan 954.3 nan
    lat_match           (HUC8, start_date) float64 nan nan nan ... nan 29.5 nan
    lon_match           (HUC8, start_date) float64 nan nan nan ... -94.75 nan

In [5]:
yaml_doc = 'plot_config_1.yaml'
config_name = 'job_1'

# import configuration file for season dictionary choice
config = yaml.load(open(yaml_doc), Loader=yaml.SafeLoader)
ddict = config[config_name]

ssn = ddict['SSN']
ARDT = ddict['ARDT']
ar = ddict['AR']
region_lst = ddict['region_lst']

In [6]:
from plot_trajectory_maps import subset_data_to_plot
subset = subset_data_to_plot(ds, ARDT, ssn, ar, region=region_lst[0], basin=None, HUC8=None)
subset

<xarray.Dataset>
Dimensions:             (HUC8: 15, start_date: 592, index: 72)
Coordinates:
  * index               (index) int64 0 1 2 3 4 5 6 7 ... 65 66 67 68 69 70 71
  * start_date          (start_date) datetime64[ns] 2000-01-11 ... 2023-12-26
  * HUC8                (HUC8) object '14050002' '14040109' ... '14030001'
    time                (HUC8, start_date, index) datetime64[ns] NaT NaT ... NaT
    lon                 (HUC8, start_date, index) float64 nan nan ... nan nan
    lat                 (HUC8, start_date, index) float64 nan nan ... nan nan
    region              (HUC8) <U17 'northern_upper_CO' ... 'northern_upper_CO'
Data variables: (12/21)
    level               (HUC8, start_date, index) float64 nan nan ... nan nan
    q                   (HUC8, start_date, index) float64 nan nan ... nan nan
    u                   (HUC8, start_date, index) float64 nan nan ... nan nan
    v                   (HUC8, start_date, index) float64 nan nan ... nan nan
    w                   (HUC8, start_date, index) float64 nan nan ... nan nan
    IVT                 (HUC8, start_date, index) float64 nan nan ... nan nan
    ...                  ...
    tARget_strict       (HUC8, start_date) float64 nan nan nan ... nan nan nan
    coastal_IVT_strict  (HUC8, start_date) float64 nan nan nan ... nan nan nan
    time_match          (HUC8, start_date) object nan nan nan ... nan nan nan
    lev_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan
    lat_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan
    lon_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan

In [11]:
count_number_trajs(subset)

1001

In [11]:
AR = subset.where(subset.ar_scale == 2, drop=True)
AR

<xarray.Dataset>
Dimensions:             (HUC8: 15, start_date: 132, index: 72)
Coordinates:
  * index               (index) int64 0 1 2 3 4 5 6 7 ... 65 66 67 68 69 70 71
  * start_date          (start_date) datetime64[ns] 2000-01-25 ... 2023-12-04
  * HUC8                (HUC8) object '14050002' '14040109' ... '14030001'
    time                (HUC8, start_date, index) datetime64[ns] NaT NaT ... NaT
    lon                 (HUC8, start_date, index) float64 nan nan ... nan nan
    lat                 (HUC8, start_date, index) float64 nan nan ... nan nan
    region              (HUC8) <U17 'northern_upper_CO' ... 'northern_upper_CO'
Data variables: (12/21)
    level               (HUC8, start_date, index) float64 nan nan ... nan nan
    q                   (HUC8, start_date, index) float64 nan nan ... nan nan
    u                   (HUC8, start_date, index) float64 nan nan ... nan nan
    v                   (HUC8, start_date, index) float64 nan nan ... nan nan
    w                   (HUC8, start_date, index) float64 nan nan ... nan nan
    IVT                 (HUC8, start_date, index) float64 nan nan ... nan nan
    ...                  ...
    tARget_strict       (HUC8, start_date) float64 nan nan nan ... nan nan nan
    coastal_IVT_strict  (HUC8, start_date) float64 nan nan nan ... nan nan nan
    time_match          (HUC8, start_date) object nan nan nan ... nan nan nan
    lev_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan
    lat_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan
    lon_match           (HUC8, start_date) float64 nan nan nan ... nan nan nan

In [12]:
nHUC8s = len(AR.HUC8.values)
ndates = len(AR.start_date.values)

for 
tmp = AR.isel(HUC8=0, start_date=0)
tmp

<xarray.Dataset>
Dimensions:             (index: 72)
Coordinates:
  * index               (index) int64 0 1 2 3 4 5 6 7 ... 65 66 67 68 69 70 71
    start_date          datetime64[ns] 2000-01-25
    HUC8                <U8 '14050002'
    time                (index) datetime64[ns] NaT NaT NaT NaT ... NaT NaT NaT
    lon                 (index) float64 nan nan nan nan nan ... nan nan nan nan
    lat                 (index) float64 nan nan nan nan nan ... nan nan nan nan
    region              <U17 'northern_upper_CO'
Data variables: (12/21)
    level               (index) float64 nan nan nan nan nan ... nan nan nan nan
    q                   (index) float64 nan nan nan nan nan ... nan nan nan nan
    u                   (index) float64 nan nan nan nan nan ... nan nan nan nan
    v                   (index) float64 nan nan nan nan nan ... nan nan nan nan
    w                   (index) float64 nan nan nan nan nan ... nan nan nan nan
    IVT                 (index) float64 nan nan nan nan nan ... nan nan nan nan
    ...                  ...
    tARget_strict       float64 nan
    coastal_IVT_strict  float64 nan
    time_match          object nan
    lev_match           float64 nan
    lat_match           float64 nan
    lon_match           float64 nan

In [14]:
data = tmp.lat.values
test = np.isnan(data).all()
if test == True:
    pass

True

In [ ]:
#############
### INPUT ###
#############
## list of vertical labels on the far left of the figure
left_lbl = ['Upper Yampa', 'Roaring Fork', 'Upper Gunnison', 'Upper San Juan']




In [ ]:
ARDT_lst = ['gwAR','rutzAR', 'ARscale']

for z, ARDT in enumerate(ARDT_lst):

    titlestring = [['(a)', '(b)', '(c)', '(d)'],
                   ['(e)', '(f)', '(g)', '(h)']]
    
    nrows = 6
    ncols = 2
    
    ## Use gridspec to set up a plot with a series of subplots that is
    ## n-rows by n-columns
    gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
    ## use gs[rows index, columns index] to access grids
    
    fig = plt.figure(figsize=(7.75, 10.))
    
    fname = path_to_figs + 'heatmaps_basin_WY_{0}'.format(ARDT)
    fmt = 'png'
    
    ############################
    ### PLOT AR SCALE VALUES ###
    ############################
    plt_lbl = ['NDJFMA', 'MJJASO']
    ssn_lst = ['NDJFMA', 'MJJASO']
    basin_lst = ['Upper Colorado', 'Rio Grande', 'Missouri', 'Arkansas']
    basin_id = [14, 13, 10, 11]
    
    ## Add color bar axis
    cbax = plt.subplot(gs[-1,:]) # colorbar axis
    
    for i, ssn in enumerate(ssn_lst):
        for j, basin_name in enumerate(basin_lst):
            if i == 0:
                llats = True
            elif i == 1:
                llats = False
            if j == 3:
                blons = True
            else: 
                blons = False
            print(ssn, basin_name)
            ax = fig.add_subplot(gs[j, i], projection=mapcrs)
            ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=llats, right_lats=False, bottom_lons=blons)
            ax.set_extent(ext, datacrs)
            ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
            
            if j == 0: # add column label to top row only
                ax.set_title(ssn, loc='left', fontsize=14)
            
            if i == 0: # add row labels to the far left plot
                ax.text(-0.16, 0.5, basin_name, va='bottom', ha='center',
                    rotation='vertical', rotation_mode='anchor', fontsize=13,
                    transform=ax.transAxes)
        
            ds = ds_lst[i].where(ds_lst[i].basin==basin_name, drop=True)
            if ARDT == 'gwAR':
                AR = ds.where(ds.tARget > 0, drop=True)
            elif ARDT == 'rutzAR':
                AR = ds.where(ds.ar > 0, drop=True)
            elif ARDT == 'ARscale':
                AR = ds.where(ds.ar_scale > 0, drop=True)
            ## now calculate heatmaps from remaining trajectories
            cell = calculate_heatmaps_from_trajectories(AR, normalize=False)
    
            ## create segmented cmap
            # cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 1.1, .1))
            cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 110, 10))
            ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
            cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                          legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})
    
    
            ## add in four basins
            plot_poly = tmp[(tmp.basin == basin_id[j])]
            plot_poly.crs = 'epsg:3857'
            print(plot_poly.crs)
            plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99)
    
            ax.text(0.03, 0.96, titlestring[i][j], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)
    
    
    
    fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
    plt.show()
    fig.clf()

In [ ]:
nrows = 4
ncols = 2

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(8.0, 5.5))
fig.dpi = 600
fname = path_to_figs + 'heatmaps_basin_WY_subset'
fmt = 'png'

############################
### PLOT AR SCALE VALUES ###
############################
plt_lbl = ['NDJFMA', 'MJJASO']
colors = ['#0ac1ff', '#04ff03', '#ffff03', '#ffa602', '#ff0100']
left_lats = [True, False]
bottom_lons = [False, True]
ssn_lst = ['NDJFMA', 'MJJASO']
basin_lst = ['Upper Colorado', 'Missouri']
basin_id = [14, 10]

## Add color bar axis
cbax = plt.subplot(gs[-1,:]) # colorbar axis

for i, ssn in enumerate(ssn_lst):
    for j, basin_name in enumerate(basin_lst):
        print(ssn, basin_name)
        ax = fig.add_subplot(gs[i, j], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=left_lats[j], right_lats=False, bottom_lons=bottom_lons[i])
        ax.set_extent(ext, datacrs)
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
        
        if i == 0: # add column label to top row only
            ax.set_title(basin_name, loc='left', fontsize=14)
        
        if j == 0: # add row labels to the far left plot
            ax.text(-0.16, 0.5, ssn_lst[i], va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
    
        ds = ds_lst[i].where(ds_lst[i].basin==basin_name, drop=True)
        AR = ds.where(ds.ar_scale > 0, drop=True)
        ## now calculate heatmaps from remaining trajectories
        cell = calculate_heatmaps_from_trajectories(AR, normalize=False)

        ## create segmented cmap
        # cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 1.1, .1))
        cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 110, 10))
        ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
        cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                      legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})


        ## add in four basins
        plot_poly = tmp[(tmp.basin == basin_id[j])]
        plot_poly.crs = 'epsg:3857'
        print(plot_poly.crs)
        plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99)


fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()

In [ ]:
nrows = 6
ncols = 4

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1, 1, 1], wspace=0.01, hspace=0.05)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(15.0, 10.0))
fig.dpi = 600
fname = path_to_figs + 'heatmaps_basin_SSN'
fmt = 'png'

############################
### PLOT AR SCALE VALUES ###
############################
plt_lbl = ['DJF', 'MAM', 'JJA', 'SON']
colors = ['#0ac1ff', '#04ff03', '#ffff03', '#ffa602', '#ff0100']
left_lats = [True, False, False, False]
bottom_lons = [False, False, False, True]
ssn_lst = ['DJF', 'MAM', 'JJA', 'SON']
basin_lst = ['Upper Colorado', 'Rio Grande', 'Missouri', 'Arkansas']
basin_id = [14, 13, 10, 11]
titlestring = [['(a)', '(b)', '(c)', '(d)'],
               ['(e)', '(f)', '(g)', '(h)'],
               ['(i)', '(j)', '(k)', '(l)'],
               ['(m)', '(n)', '(o)', '(p)']]
## Add color bar
cbax = plt.subplot(gs[-1,:]) # colorbar axis

for i, ssn in enumerate(ssn_lst):
    for j, basin_name in enumerate(basin_lst):
        print(ssn, basin_name)
        ax = fig.add_subplot(gs[i, j], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=left_lats[j], right_lats=False, bottom_lons=bottom_lons[i])
        ax.set_extent(ext, datacrs)
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
        
        if i == 0: # add column label to top row only
            ax.set_title(basin_name, loc='left', fontsize=14)
        
        if j == 0: # add row labels to the far left plot
            ax.text(-0.16, 0.5, ssn_lst[i], va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
    
        ds = ds_lst2[i].where(ds_lst2[i].basin==basin_name, drop=True)
        AR = ds.where(ds.ar_scale > 0, drop=True)
        ## now calculate heatmaps from remaining trajectories
        cell = calculate_heatmaps_from_trajectories(AR, normalize=False)
        # cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 1.1, .1))
        cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 110, 10))
        ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
        cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                      legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})


        ## add in four basins
        plot_poly = tmp[(tmp.basin == basin_id[j])]
        plot_poly.crs = 'epsg:3857'
        print(plot_poly.crs)
        plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99)
        ax.text(0.03, 0.96, titlestring[i][j], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)



fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()

In [ ]:
titlestring = [['(a)', '(b)', '(c)', '(d)'],
               ['(e)', '(f)', '(g)', '(h)']]

nrows = 6
ncols = 2

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(7.75, 10.))
fname = path_to_figs + 'heatmaps_basin_WY_nonAR'
fmt = 'png'

############################
### PLOT AR SCALE VALUES ###
############################
plt_lbl = ['NDJFMA', 'MJJASO']
ssn_lst = ['NDJFMA', 'MJJASO']
basin_lst = ['Upper Colorado', 'Rio Grande', 'Missouri', 'Arkansas']
basin_id = [14, 13, 10, 11]

## Add color bar axis
cbax = plt.subplot(gs[-1,:]) # colorbar axis

for i, ssn in enumerate(ssn_lst):
    for j, basin_name in enumerate(basin_lst):
        if i == 0:
            llats = True
        elif i == 1:
            llats = False
        if j == 3:
            blons = True
        else: 
            blons = False
        print(ssn, basin_name)
        ax = fig.add_subplot(gs[j, i], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=llats, right_lats=False, bottom_lons=blons)
        ax.set_extent(ext, datacrs)
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
        
        if j == 0: # add column label to top row only
            ax.set_title(ssn, loc='left', fontsize=14)
        
        if i == 0: # add row labels to the far left plot
            ax.text(-0.16, 0.5, basin_name, va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
    
        ds = ds_lst[i].where(ds_lst[i].basin==basin_name, drop=True)
        AR = ds.where(ds.ar_scale.isnull(), drop=True)
        ## now calculate heatmaps from remaining trajectories
        cell = calculate_heatmaps_from_trajectories(AR, normalize=False, AR=False)

        ## create segmented cmap
        # cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 1.1, .1))
        cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 425, 25))
        ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
        cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                      legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})


        ## add in four basins
        plot_poly = tmp[(tmp.basin == basin_id[j])]
        plot_poly.crs = 'epsg:3857'
        print(plot_poly.crs)
        plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99)

        ax.text(0.03, 0.96, titlestring[i][j], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)



fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()

In [ ]:
nrows = 4
ncols = 1

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 0.05, 0.05], width_ratios = [1], wspace=0.01, hspace=0.05)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(6., 8.))
fig.dpi = 600
fname = path_to_figs + 'heatmaps_all_WY'
fmt = 'png'

############################
### PLOT AR SCALE VALUES ###
############################
plt_lbl = ['NDJFMA', 'MJJASO']
bottom_lons = [False, True]
ssn_lst = ['NDJFMA', 'MJJASO']
cbax = plt.subplot(gs[-1,:]) # colorbar axis


for i, ssn in enumerate(ssn_lst):
    print(ssn)
    ax = fig.add_subplot(gs[i, 0], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=True, right_lats=False, bottom_lons=bottom_lons[i])
    ax.set_extent(ext, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)

    # add row labels to the far left plot
    ax.text(-0.16, 0.5, ssn_lst[i], va='bottom', ha='center',
        rotation='vertical', rotation_mode='anchor', fontsize=13,
        transform=ax.transAxes)

    ds = ds_lst[i]
    AR = ds.where(ds.ar_scale > 0, drop=True)
    ## now calculate heatmaps from remaining trajectories
    cell = calculate_heatmaps_from_trajectories(AR, normalize=False)

    ## create segmented cmap
    # cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 1.1, .1))
    cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 220, 10))
    ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
    cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                  legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})


    ## add in four basins
    plot_poly = tmp
    plot_poly.crs = 'epsg:3857'
    print(plot_poly.crs)
    plot_poly.plot(ax=ax, edgecolor='gray', color='None', zorder=99)

fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()